# BARAM 2026 — FICR-정렬 손실 가중 + MLP 재튜닝 (v8 후보)

v7(LB 0.63497, 139위) 이후. FICR 갭(0.399 vs 상위 0.43+)을 학습 단계에서 직접 겨냥.

- **1부: FICR-정렬 가중** — FICR은 `Σ(실측×단가)/Σ(실측×4)`로 발전량 가중이므로, 학습 손실에도
  `w = stuck가중 × (1 + α·y/cap)` 를 걸어 고출력 시간 정확도를 우대. α ∈ {0.5, 1, 2} vs 현행(α=0).
- **2부: MLP 재튜닝** — 채택 α 하에서 12 trial random search (GBM 예측은 1부 캐시 재사용, MLP만 재학습).
- 판정: 두 폴드(2023·2024) 모두 현행 이상. 현행 참조: raw 0.6098/0.6176.
- 캘리브레이션: LB ≈ holdout ±0.002 (v6·v7 실측).

## 0. 공통 설정

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS","1")
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
torch.set_num_threads(1)
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
import wind_lib as W
from official_metric import group_scores

DEV="mps" if torch.backends.mps.is_available() else "cpu"
SEEDS=[15,0,1]; BLEND=0.7; STUCK_TH=0.3; STUCK_W=0.5
MLP_CFG=dict(h=256,depth=3,drop=0.0,lr=0.0015868563457489381,wd=1e-4,bs=256,emb=4)
GBM_PARAMS=dict(objective="mae",n_estimators=2000,learning_rate=0.020651822836313095,
    num_leaves=63,min_child_samples=300,subsample=0.8,subsample_freq=1,colsample_bytree=0.5,
    reg_lambda=0.1,random_state=42,n_jobs=1,verbose=-1)
RAW=os.environ.get("WIND_RAW_DIR",os.path.expanduser("~/Downloads/open"))
GROUPS=(1,2,3)
FR,TGT={},{}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=W.add_spatial(W.build(df,g),"train")
BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS]+["pc_pred_cf"]
FEATS=W.lean_features(BASE_ALL)+W.SPATIAL_COLS

def stuck_frac():
    frames={}
    for fn,pre,n,rate in [("scada_vestas_train.csv","vestas",12,3600.0),
                          ("scada_unison_train.csv","unison",5,4200.0)]:
        d=pd.read_csv(f"{RAW}/train/{fn}",encoding="utf-8-sig",parse_dates=["kst_dtm"])
        d["hour"]=d["kst_dtm"].dt.ceil("h")
        agg={}
        for i in range(1,n+1):
            pw=f"{pre}_wtg{i:02d}_power_kw10m"; ws=f"{pre}_wtg{i:02d}_ws"
            h=d.groupby("hour").agg(pw_m=(pw,"mean"),ws_m=(ws,"mean"))
            agg[i]=pd.DataFrame({f"stuck_{i}":((h.ws_m>=4.0)&(h.pw_m<=0.01*rate)).astype(float),
                                 f"rep_{i}":h.pw_m.notna().astype(float)})
        frames[pre]=pd.concat(agg.values(),axis=1)
    def frac(pre,ids):
        f=frames[pre]
        st=f[[f"stuck_{i}" for i in ids]].sum(axis=1); rp=f[[f"rep_{i}" for i in ids]].sum(axis=1)
        return (st/rp).where(rp>=3)
    return {1:frac("vestas",range(1,7)),2:frac("vestas",range(7,13)),3:frac("unison",range(1,6))}
FRAC=stuck_frac()
for g in GROUPS:
    s=FRAC[g].reindex(FR[g].kst_dtm).to_numpy()
    FR[g]=FR[g].assign(stuck_frac=np.nan_to_num(s,nan=0.0))

def make_weight(fr, tgt, cap, alpha):
    """학습 가중: stuck × FICR-정렬."""
    w=np.where(fr["stuck_frac"]>=STUCK_TH, STUCK_W, 1.0)
    if alpha>0:
        w=w*(1.0+alpha*np.clip(fr[tgt].to_numpy()/cap,0,1))
    return w

class MLP(nn.Module):
    def __init__(s,nf,ng=3,h=256,depth=3,drop=0.0,emb=4):
        super().__init__(); s.emb=nn.Embedding(ng,emb)
        L=[nn.Linear(nf+emb,h),nn.GELU(),nn.Dropout(drop)]
        for _ in range(depth-1): L+=[nn.Linear(h,h),nn.GELU(),nn.Dropout(drop)]
        L+=[nn.Linear(h,1)]; s.net=nn.Sequential(*L)
    def forward(s,x,g): return s.net(torch.cat([x,s.emb(g)],-1)).squeeze(-1)

def train_one(pool_tr,seed,cfg=None):
    cfg=cfg or MLP_CFG
    torch.manual_seed(seed); np.random.seed(seed)
    pool_tr=pool_tr.sort_values("kst_dtm")
    mu=pool_tr[FEATS].mean(); sd=pool_tr[FEATS].std()+1e-8
    X=((pool_tr[FEATS]-mu)/sd).to_numpy(np.float32)
    y=pool_tr["cf"].to_numpy(np.float32); gid=pool_tr["gid"].to_numpy(np.int64)
    wt=pool_tr["w"].to_numpy(np.float32)
    n=len(X); cut=int(n*0.85)
    Xt=torch.tensor(X,device=DEV); yt=torch.tensor(y,device=DEV)
    gt=torch.tensor(gid,device=DEV); wtt=torch.tensor(wt,device=DEV)
    net=MLP(len(FEATS),h=cfg["h"],depth=cfg["depth"],drop=cfg["drop"],emb=cfg["emb"]).to(DEV)
    opt=torch.optim.AdamW(net.parameters(),lr=cfg["lr"],weight_decay=cfg["wd"])
    best=1e9; st=None; pat=0
    for ep in range(120):
        net.train(); perm=np.random.permutation(np.arange(cut))
        for i in range(0,len(perm),cfg["bs"]):
            b=torch.tensor(perm[i:i+cfg["bs"]],device=DEV); opt.zero_grad()
            e=(net(Xt[b],gt[b])-yt[b]).abs()
            ((e*wtt[b]).sum()/(wtt[b].sum()+1e-8)).backward(); opt.step()
        net.eval()
        with torch.no_grad():
            e=(net(Xt[cut:],gt[cut:])-yt[cut:]).abs()
            vl=((e*wtt[cut:]).sum()/(wtt[cut:].sum()+1e-8)).item()
        if vl<best-1e-5: best=vl; st={k:v.clone() for k,v in net.state_dict().items()}; pat=0
        else: pat+=1
        if pat>=10: break
    net.load_state_dict(st); return net,(mu,sd)

def predict_one(net,scaler,fr,g,cap):
    mu,sd=scaler
    X=torch.tensor(((fr[FEATS]-mu)/sd).to_numpy(np.float32),device=DEV)
    gid=torch.full((len(fr),),g-1,dtype=torch.long,device=DEV)
    net.eval()
    with torch.no_grad(): p=net(X,gid).cpu().numpy()
    return np.clip(p,0,1)*cap

FOLDS={2023:[2022],2024:[2022,2023]}
CACHE={}
for vy,tys in FOLDS.items():
    ent={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
        tr=fr[yr.isin(tys)]; va=fr[yr==vy]
        if len(tr)==0 or len(va)==0: continue
        iso=W.fit_powercurve(tr,tgt,cap)
        ent[g]=(W.with_pc(tr,iso),W.with_pc(va,iso))
    CACHE[vy]=ent

def totals_from(gbm,mlp,vy,B=BLEND):
    nm=[]; fi=[]
    for g,(_,va2) in CACHE[vy].items():
        cap=W.CAP[g]
        p=np.clip((1-B)*gbm[g]+B*mlp[g],0,cap)
        a,b=group_scores(va2[TGT[g]].to_numpy(),p,cap); nm.append(a); fi.append(b)
    return 0.5*(1-np.mean(nm))+0.5*np.mean(fi)
print("ready")

ready


## 1부. FICR-정렬 가중 α 스캔

In [2]:
GBM_CACHE={}; MLP_CACHE={}
def run_alpha(alpha, mlp_cfg=None, seeds=SEEDS):
    out={}
    for vy,ent in CACHE.items():
        # GBM (α별 캐시)
        key=(vy,alpha)
        if key not in GBM_CACHE:
            gbm={}
            for g,(tr2,va2) in ent.items():
                cap=W.CAP[g]; tgt=TGT[g]; w=make_weight(tr2,tgt,cap,alpha)
                lg_=lgb.LGBMRegressor(**GBM_PARAMS).fit(tr2[FEATS],tr2[tgt],sample_weight=w)
                hg_=HGB(loss="absolute_error",max_iter=600,learning_rate=0.03,max_leaf_nodes=63,
                    l2_regularization=1.0,random_state=42).fit(tr2[FEATS].to_numpy(),tr2[tgt].to_numpy(),sample_weight=w)
                gbm[g]=np.clip(0.5*(lg_.predict(va2[FEATS])+hg_.predict(va2[FEATS].to_numpy())),0,W.CAP[g])
            GBM_CACHE[key]=gbm
        # MLP
        pool=[]
        for g,(tr2,_) in ent.items():
            p=tr2[FEATS+["kst_dtm"]].copy()
            p["cf"]=tr2[TGT[g]]/W.CAP[g]; p["gid"]=g-1
            p["w"]=make_weight(tr2,TGT[g],W.CAP[g],alpha); pool.append(p)
        pool=pd.concat(pool,ignore_index=True)
        acc={g:[] for g in ent}
        for sd_ in seeds:
            net,scaler=train_one(pool,sd_,mlp_cfg)
            for g,(_,va2) in ent.items():
                acc[g].append(predict_one(net,scaler,va2,g,W.CAP[g]))
        mlp={g:np.mean(v,axis=0) for g,v in acc.items()}
        MLP_CACHE[(vy,alpha,"base")]=mlp
        out[vy]=totals_from(GBM_CACHE[key],mlp,vy)
    return out

res_a={}
for alpha in [0.0,0.5,1.0,2.0]:
    res_a[alpha]=run_alpha(alpha)
    r=res_a[alpha]
    print(f"α={alpha}: 2023={r[2023]:.4f}  2024={r[2024]:.4f}  min={min(r.values()):.4f}")

cur=res_a[0.0]
cands=[a for a in [0.5,1.0,2.0] if res_a[a][2023]>=cur[2023] and res_a[a][2024]>=cur[2024]]
ALPHA=max(cands,key=lambda a:min(res_a[a].values())) if cands else 0.0
print(f"\n채택 α = {ALPHA}" + ("" if ALPHA>0 else " (개선 없음 → 가중 미적용)"))

α=0.0: 2023=0.6098  2024=0.6176  min=0.6098


α=0.5: 2023=0.6109  2024=0.6204  min=0.6109


α=1.0: 2023=0.6152  2024=0.6239  min=0.6152


α=2.0: 2023=0.6173  2024=0.6241  min=0.6173

채택 α = 2.0


## 2부. MLP 재튜닝 (채택 α, GBM 캐시 재사용)

In [3]:
rng=np.random.RandomState(11)
def sample_cfg():
    return dict(h=int(rng.choice([128,256,384])), depth=int(rng.choice([2,3,4])),
                drop=float(rng.choice([0.0,0.1,0.2])),
                lr=float(10**rng.uniform(np.log10(5e-4),np.log10(3e-3))),
                wd=float(rng.choice([1e-5,1e-4,1e-3])),
                bs=int(rng.choice([256,512])), emb=int(rng.choice([4,8])))

base23,base24=res_a[ALPHA][2023],res_a[ALPHA][2024]
print(f"기준(현행 MLP_CFG, α={ALPHA}): 2023={base23:.4f} 2024={base24:.4f}")
trials=[]
for t in range(12):
    cfg=sample_cfg(); tot={}
    for vy,ent in CACHE.items():
        pool=[]
        for g,(tr2,_) in ent.items():
            p=tr2[FEATS+["kst_dtm"]].copy()
            p["cf"]=tr2[TGT[g]]/W.CAP[g]; p["gid"]=g-1
            p["w"]=make_weight(tr2,TGT[g],W.CAP[g],ALPHA); pool.append(p)
        pool=pd.concat(pool,ignore_index=True)
        acc={g:[] for g in ent}
        for sd_ in SEEDS:
            net,scaler=train_one(pool,sd_,cfg)
            for g,(_,va2) in ent.items():
                acc[g].append(predict_one(net,scaler,va2,g,W.CAP[g]))
        mlp={g:np.mean(v,axis=0) for g,v in acc.items()}
        tot[vy]=totals_from(GBM_CACHE[(vy,ALPHA)],mlp,vy)
    ok=(tot[2023]>=base23) and (tot[2024]>=base24)
    trials.append((t,cfg,tot,ok))
    print(f"trial {t:2d}: 23={tot[2023]:.4f} 24={tot[2024]:.4f} min={min(tot.values()):.4f} "
          f"h={cfg['h']} d={cfg['depth']} drop={cfg['drop']} lr={cfg['lr']:.4f} beats={ok}")

cands=[x for x in trials if x[3]]
if cands:
    t,bc,bt,_=max(cands,key=lambda x:min(x[2].values()))
    print(f"\nMLP 재튜닝 채택 trial {t}: 23={bt[2023]:.4f} 24={bt[2024]:.4f}")
    NEW_MLP=bc
else:
    print("\n현행 MLP_CFG를 두 해 모두 이기는 설정 없음 → 유지")
    NEW_MLP=None; bt={2023:base23,2024:base24}

summary=dict(alpha_scan={str(a):{"2023":round(r[2023],4),"2024":round(r[2024],4)} for a,r in res_a.items()},
    adopted_alpha=ALPHA,
    mlp_retune={"adopted":(NEW_MLP is not None),"cfg":NEW_MLP,
                "totals":{"2023":round(bt[2023],4),"2024":round(bt[2024],4)}},
    v7_reference={"2023":0.6098,"2024":0.6176})
json.dump(summary,open("submission/ver_8/ficrw_result.json","w"),ensure_ascii=False,indent=2)
print(json.dumps(summary,ensure_ascii=False,indent=2))

기준(현행 MLP_CFG, α=2.0): 2023=0.6173 2024=0.6241


trial  0: 23=0.6051 24=0.6294 min=0.6051 h=256 d=2 drop=0.1 lr=0.0007 beats=False


trial  1: 23=0.6149 24=0.6236 min=0.6149 h=384 d=2 drop=0.1 lr=0.0025 beats=False


trial  2: 23=0.6129 24=0.6287 min=0.6129 h=128 d=3 drop=0.0 lr=0.0018 beats=False


trial  3: 23=0.6048 24=0.6281 min=0.6048 h=256 d=2 drop=0.1 lr=0.0016 beats=False


...

trial  7: 23=0.6080 24=0.6236 min=0.6080 h=384 d=3 drop=0.0 lr=0.0012 beats=False


trial  8: 23=0.6074 24=0.6308 min=0.6074 h=384 d=2 drop=0.2 lr=0.0006 beats=False


trial  9: 23=0.6131 24=0.6263 min=0.6131 h=256 d=4 drop=0.1 lr=0.0008 beats=False


trial 10: 23=0.6109 24=0.6256 min=0.6109 h=128 d=2 drop=0.0 lr=0.0022 beats=False


trial 11: 23=0.6002 24=0.6280 min=0.6002 h=384 d=2 drop=0.0 lr=0.0016 beats=False

현행 MLP_CFG를 두 해 모두 이기는 설정 없음 → 유지
{
  "alpha_scan": {
    "0.0": {
      "2023": 0.6098,
      "2024": 0.6176
    },
    "0.5": {
      "2023": 0.6109,
      "2024": 0.6204
    },
    "1.0": {
      "2023": 0.6152,
      "2024": 0.6239
    },
    "2.0": {
      "2023": 0.6173,
      "2024": 0.6241
    }
  },
  "adopted_alpha": 2.0,
  "mlp_retune": {
    "adopted": false,
    "cfg": null,
    "totals": {
      "2023": 0.6173,
      "2024": 0.6241
    }
  },
  "v7_reference": {
    "2023": 0.6098,
    "2024": 0.6176
  }
}


## 결론
- α>0 채택 + MLP 재튜닝 채택 시 → v8 파이프라인(시드 10)으로 재구축해 holdout 확인(≈LB) 후 제출.
- 둘 다 기각이면 v7 유지가 확정 — 이 구성 계열의 레버 소진.